# MPC and MPPI

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/advanced/03_mpc_and_mppi.ipynb)

Close the loop around a planner: model predictive control, a sampling planner in its place, and its weights tuned by gradient.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. A planner in the loop

An open-loop plan drifts off an unstable target (see `02_optimal_control`). Model
predictive control re-plans at every tick from the measured state and applies only the
start of each plan:

$$u_k = u^\star_0(x_k), \qquad u^\star = \arg\min_{u(\cdot)} \int_{t_k}^{t_k + H} g(x, u)\, dt + h(x(t_k + H)).$$

The task is the pendulum swing-up of `02_optimal_control`, scored over 6 s. The plan is a
shorter problem of its own: a 3 s horizon $H$ with a terminal weight $S$ that stands for
what comes after it.

In [ ]:
import numpy as np

from minilink import BallSet, Pendulum, PlanningProblem, QuadraticCost

pendulum = Pendulum()
x_up = np.array([np.pi, 0.0])

score = QuadraticCost.from_system(pendulum, x_bar=x_up, Q=np.eye(2), R=0.1 * np.eye(1))
problem = PlanningProblem(
    pendulum,
    x0=np.zeros(2),
    cost=score,
    U=pendulum.input_box,
    Xf=BallSet(x_up, 0.3),
    horizon=6.0,
)

plan_cost = QuadraticCost.from_system(
    pendulum, x_bar=x_up, Q=np.eye(2), R=0.1 * np.eye(1), S=10.0 * np.eye(2)
)
plan_problem = PlanningProblem(
    pendulum, x0=np.zeros(2), cost=plan_cost, U=pendulum.input_box, horizon=3.0
)

`ModelPredictiveController` wraps a planner into a controller block. Each solve starts from
the last plan shifted by one tick (the warm start, on by default), so a few solver
iterations per tick are enough. `delay=True` applies at tick $k$ the plan solved at tick
$k - 1$, as on a real computer where the solve takes time.

`(mpc % dt) @ plant` is the usual sampled loop: the MPC ticks at 10 Hz, the plant
integrates between ticks.

In [ ]:
from minilink import TrajectoryOptimizationPlanner
from minilink.control import ModelPredictiveController

planner = TrajectoryOptimizationPlanner(
    plan_problem,
    n_steps=30,
    transcription="direct_collocation",
    backend="jax",
    method="slsqp",
    options={"maxiter": 100},
)
mpc = ModelPredictiveController(planner, delay=True)

pendulum.x0 = np.zeros(2)
loop = (mpc % 0.1) @ pendulum
traj = loop.compute_trajectory(tf=6.0)

The loop's state holds the plan in the MPC's memory, so the plot reads the plant's part
of the run, `traj.of("plant")`.

In [ ]:
pendulum.plot_trajectory(traj.of("plant"))

The MPC is a controller like any other, so the problem scores it the usual way:
`problem.compile(mpc % dt).evaluate()` runs the loop and reports its cost and whether it
respects `U` and reaches `Xf`. Unlike the replayed plans of the previous notebook, it ends
in the goal.

In [ ]:
print(problem.compile(mpc % 0.1).evaluate())

## 2. Signals into the plan: `inputs=`

By default the measurement on port `y` becomes the plan's start, `{"x0": "y"}`. Any
other number of the plan can be bound to an input port the same way. Here the cost's target
`cost.x_bar` reads a port `r`: the MPC now tracks a reference signal. The pendulum rests at
the bottom until the reference steps to the upright at $t = 1$ s; the delay shows as one
tick of lag.

In [ ]:
from minilink import Step, closed_loop

tracking_mpc = ModelPredictiveController(
    planner, delay=True, inputs={"x0": "y", "cost.x_bar": "r"}
)
print(tracking_mpc)

reference = Step(x_up, np.zeros(2), step_time=1.0)
tracking_loop = closed_loop(tracking_mpc % 0.1, pendulum, reference=reference)
tracking_traj = tracking_loop.compute_trajectory(tf=6.0)
pendulum.plot_trajectory(tracking_traj.of("plant"))

## 3. MPPI: a sampling planner in the same loop

`MPPIPlanner` (model predictive path integral) plans without gradients: at each solve it
draws $K$ noisy input sequences around the last plan, simulates them all, and averages them
weighted by $e^{-J_i / \lambda}$. It needs no smooth cost, but it needs one that guides the
samples: the wrapped angle error, so that both ways up are equally good.

In [ ]:
import jax.numpy as jnp

from minilink import CostFunction


class SwingUpCost(CostFunction):
    def g(self, x, u, t=0.0, params=None):
        theta = x[0]
        dtheta = x[1]

        # The angle to the upright, wrapped to (-π, π]
        e = jnp.arctan2(jnp.sin(theta - jnp.pi), jnp.cos(theta - jnp.pi))

        g = e**2 + 0.1 * dtheta**2 + 0.001 * u[0] ** 2

        return g

    def h(self, x, t=0.0, params=None):
        return 0.0

The planner drops into the same controller block: 512 samples over a 2 s horizon at
20 Hz.

In [ ]:
from minilink.planning import MPPIPlanner

mppi_problem = PlanningProblem(
    pendulum, x0=np.zeros(2), cost=SwingUpCost(), U=pendulum.input_box, horizon=2.0
)
mppi = MPPIPlanner(mppi_problem, dt=0.05, n_rollouts=512, temperature=0.1, noise=9.0)
mppi_controller = ModelPredictiveController(mppi)

mppi_loop = (mppi_controller % 0.05) @ pendulum
mppi_traj = mppi_loop.compute_trajectory(tf=6.0, dt=0.05)
pendulum.plot_trajectory(mppi_traj.of("plant"))

Scored on the same problem, the sampled plan pays more than the optimized one: it pumps
longer before it reaches the top.

In [ ]:
print(problem.compile(mppi_controller % 0.05).evaluate())

## 4. Tuning an MPC by gradient

The MPC's weights $Q$ and $R$ are parameters of the loop like any other: the loop's
`controller.solve.cost.Q` is the plan's $Q$. To differentiate the scored cost with respect
to them, every tick must run a fixed computation that JAX can trace: a native planner
with a fixed number of Adam iterations instead of SciPy's solver.

The task: bring the pendulum down from 1 rad with little effort, scored over 3 s, with a
1 s plan whose weights start badly tuned.

In [ ]:
task_cost = QuadraticCost.from_system(
    pendulum, Q=np.diag([1.0, 0.1]), R=0.01 * np.eye(1)
)
task = PlanningProblem(
    pendulum, x0=np.array([1.0, 0.0]), cost=task_cost, U=pendulum.input_box, horizon=3.0
)

untuned_cost = QuadraticCost.from_system(pendulum, Q=np.eye(2), R=np.eye(1))
native_planner = TrajectoryOptimizationPlanner(
    PlanningProblem(
        pendulum,
        x0=np.array([1.0, 0.0]),
        cost=untuned_cost,
        U=pendulum.input_box,
        horizon=1.0,
    ),
    n_steps=10,
    transcription="single_shooting",
    backend="jax",
    method="adam",
    n_iterations=30,
    learning_rate=0.05,
)
native_mpc = ModelPredictiveController(native_planner, delay=True)

evaluator = task.compile(native_mpc % 0.1, substeps=1, backend="jax")
start = evaluator.params
print(evaluator.evaluate(start))

`evaluator.program(wrt)` is the scored cost as a function of the chosen numbers, with its
gradient through every tick and every Adam step; `Optimizer` minimizes it.

In [ ]:
from minilink import Optimizer

wrt = ["controller.solve.cost.Q", "controller.solve.cost.R"]
best = Optimizer(evaluator.program(wrt, params=start), method="auto").solve()
tuned = evaluator.unpack(best.z, wrt, params=start)
print(evaluator.evaluate(tuned))

In [ ]:
pendulum.plot_trajectory(evaluator.trajectory(tuned).of("plant"))

## 5. A bigger plant: a car

Nothing above is special to the pendulum. The dynamic bicycle `BicycleDynRate` has eight
states (the pose, the body velocities, the rear-wheel rate and the steering angle) and two
inputs, the wheel's angular acceleration and the steering rate, so the actuators' limits are
bounds of the plan. The task: from 3 m off the lane and below the cruise speed, come back
onto the line $y = 0$ at 4 m/s.

The loop is the same three lines: a planner on a short horizon, the controller block, the
clock. Five collocation intervals over 2 s and ten solver iterations per tick are enough,
because each solve starts from the last plan.

In [ ]:
from minilink import BicycleDynRate

car = BicycleDynRate()
v_cruise = 4.0  # [m/s]
w_cruise = v_cruise / car.params["r_r"]  # the rear wheel's rate at that speed [rad/s]

x_lane = np.array([0.0, 0.0, 0.0, v_cruise, 0.0, 0.0, w_cruise, 0.0])
car.x0 = np.array([0.0, 3.0, 0.0, 0.8 * v_cruise, 0.0, 0.0, 0.8 * w_cruise, 0.0])

lane_cost = QuadraticCost.from_system(
    car,
    Q=np.diag([0.0, 12.0, 18.0, 0.5, 4.0, 6.0, 0.1, 100.0]),  # x is free: the car drives on
    R=np.diag([1.0, 25.0]),
    S=np.diag([0.0, 30.0, 40.0, 2.0, 12.0, 18.0, 0.1, 100.0]),
    x_bar=x_lane,
)
lane_problem = PlanningProblem(car, x0=car.x0, cost=lane_cost, horizon=2.0)
lane_planner = TrajectoryOptimizationPlanner(
    lane_problem,
    n_steps=5,
    transcription="direct_collocation",
    backend="jax",
    method="slsqp",
    options={"maxiter": 10, "ftol": 1.0},
)

car_loop = (ModelPredictiveController(lane_planner) % 0.05) @ car
car_traj = car_loop.compute_trajectory(tf=5.0, dt=0.01)
car.plot_trajectory(car_traj.of("plant"))
car_traj.of("plant").x[:, -1].round(2)

In [ ]:
car.animate(car_traj.of("plant"))

After 5 s the car is on the line, within a centimetre, at 3.9 m/s. The scripts under
`examples/demos/` take the same loop to a closed circuit with cones, and to the 1/10
racecar.

## Summary

| Step | Code |
| --- | --- |
| A planner as a controller | `mpc = ModelPredictiveController(planner, delay=True)` |
| The sampled loop | `(mpc % dt) @ plant` |
| Scored on a problem | `problem.compile(mpc % dt).evaluate()` |
| Signals into the plan | `inputs={"x0": "y", "cost.x_bar": "r"}` |
| Sampling instead of optimizing | `ModelPredictiveController(MPPIPlanner(...))` |
| The same loop on a car | `(ModelPredictiveController(planner) % dt) @ BicycleDynRate()` |
| Tuning the weights | native planner (`method="adam"`, `n_iterations=`), then `Optimizer(evaluator.program(wrt))` |

Next, [`04_reinforcement_learning`](04_reinforcement_learning.ipynb): a policy learned
from simulated experience.